In [1]:
!pip install scikit-learn



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
!pip install sentence-transformers

  Using cached sentence_transformers-5.3.0-py3-none-any.whl.metadata (16 kB)
Using cached sentence_transformers-5.3.0-py3-none-any.whl (512 kB)



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import pandas as pd
import re
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

c:\Users\Hemanth Sai\OneDrive\Desktop\info\queytube\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
df = pd.read_csv("video_with_transcripts.csv")

print("Dataset Shape:", df.shape)

df.head()

Dataset Shape: (350, 7)


,video_id,title,publish_date,year,month,title_length,transcript
0,cw02dMpWStI,How to find good open source projects to contr...,2026-03-04 13:29:25+00:00,2026,3,76,when I was in I was working I was probably wor...
1,0WjfKQdfeMU,NVIDIA-Certified Associate AI Infrastructure a...,2026-03-04 11:00:02+00:00,2026,3,88,"Hey, this is Andrew Brown bringing you another..."
2,ZldNCx4AvEM,Learn the basics of Git in 60 seconds with Bea...,2026-03-03 13:05:25+00:00,2026,3,54,Let's learn the basics of Git in less than 60 ...
3,E7o_WdfKszU,There's so much to learn - so how do you focus...,2026-03-02 13:17:28+00:00,2026,3,79,And you know what one thing I will say is I th...
4,IBTx5aGj-6U,Build Your Own Video Sharing App – Loom Clone ...,2026-03-02 11:00:33+00:00,2026,3,86,"In this course, we're going to build a fully f..."


In [7]:
required_cols = ["video_id", "title", "publish_date", "transcript"]

for col in required_cols:
    if col not in df.columns:
        print(f"{col} column missing ❌")
    else:
        print(f"{col} column exists ✅")

video_id column exists ✅
title column exists ✅
publish_date column exists ✅
transcript column exists ✅


In [8]:
def clean_text(text):
    
    if pd.isna(text):
        return text
    
    text = str(text)

    # remove newline and tab
    text = re.sub(r'\n|\t', ' ', text)

    # remove special characters
    text = re.sub(r'[^\w\s]', '', text)

    # remove multiple spaces
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

In [9]:
df["title"] = df["title"].apply(clean_text)

df["transcript"] = df["transcript"].apply(clean_text)

df.head()

,video_id,title,publish_date,year,month,title_length,transcript
0,cw02dMpWStI,How to find good open source projects to contr...,2026-03-04 13:29:25+00:00,2026,3,76,when I was in I was working I was probably wor...
1,0WjfKQdfeMU,NVIDIACertified Associate AI Infrastructure an...,2026-03-04 11:00:02+00:00,2026,3,88,Hey this is Andrew Brown bringing you another ...
2,ZldNCx4AvEM,Learn the basics of Git in 60 seconds with Bea...,2026-03-03 13:05:25+00:00,2026,3,54,Lets learn the basics of Git in less than 60 s...
3,E7o_WdfKszU,Theres so much to learn so how do you focus to...,2026-03-02 13:17:28+00:00,2026,3,79,And you know what one thing I will say is I th...
4,IBTx5aGj-6U,Build Your Own Video Sharing App Loom Clone wi...,2026-03-02 11:00:33+00:00,2026,3,86,In this course were going to build a fully fun...


In [10]:
print("Null transcripts:", df["transcript"].isnull().sum())

Null transcripts: 5


In [11]:
df = df.dropna(subset=["transcript"])

In [12]:
df = df[df["transcript"].str.strip() != ""]

In [13]:
df = df.rename(columns={"publish_date": "datetime"})

In [14]:
df["datetime"] = pd.to_datetime(df["datetime"])

In [15]:
df = df[["video_id", "title", "datetime", "transcript"]]

In [16]:
df.head()

,video_id,title,datetime,transcript
0,cw02dMpWStI,How to find good open source projects to contr...,2026-03-04 13:29:25+00:00,when I was in I was working I was probably wor...
1,0WjfKQdfeMU,NVIDIACertified Associate AI Infrastructure an...,2026-03-04 11:00:02+00:00,Hey this is Andrew Brown bringing you another ...
2,ZldNCx4AvEM,Learn the basics of Git in 60 seconds with Bea...,2026-03-03 13:05:25+00:00,Lets learn the basics of Git in less than 60 s...
3,E7o_WdfKszU,Theres so much to learn so how do you focus to...,2026-03-02 13:17:28+00:00,And you know what one thing I will say is I th...
4,IBTx5aGj-6U,Build Your Own Video Sharing App Loom Clone wi...,2026-03-02 11:00:33+00:00,In this course were going to build a fully fun...


In [17]:
print("Final Dataset Shape:", df.shape)

print("\nNull values:\n", df.isnull().sum())

Final Dataset Shape: (345, 4)

Null values:
 video_id      0
title         0
datetime      0
transcript    0
dtype: int64


In [18]:
df.to_csv("cleaned_transcripts.csv", index=False)

print("Dataset saved as cleaned_transcripts.csv")

Dataset saved as cleaned_transcripts.csv


In [19]:
df = pd.read_csv("cleaned_transcripts.csv")

print("Dataset Shape:", df.shape)

df.head()

Dataset Shape: (345, 4)


,video_id,title,datetime,transcript
0,cw02dMpWStI,How to find good open source projects to contr...,2026-03-04 13:29:25+00:00,when I was in I was working I was probably wor...
1,0WjfKQdfeMU,NVIDIACertified Associate AI Infrastructure an...,2026-03-04 11:00:02+00:00,Hey this is Andrew Brown bringing you another ...
2,ZldNCx4AvEM,Learn the basics of Git in 60 seconds with Bea...,2026-03-03 13:05:25+00:00,Lets learn the basics of Git in less than 60 s...
3,E7o_WdfKszU,Theres so much to learn so how do you focus to...,2026-03-02 13:17:28+00:00,And you know what one thing I will say is I th...
4,IBTx5aGj-6U,Build Your Own Video Sharing App Loom Clone wi...,2026-03-02 11:00:33+00:00,In this course were going to build a fully fun...


In [20]:
with open("queries.txt", "r") as f:
    queries = [line.strip() for line in f.readlines()]

print("Total queries:", len(queries))

queries[:10]

Total queries: 77


['How do you build a video sharing app using Next.js?',
 'What are closures in JavaScript?',
 'How do regular expressions work in programming?',
 'What is integration testing in software development?',
 'How do you contribute to open source projects?',
 'What is Git and why is it important?',
 'How do developers collaborate using GitHub?',
 'What is Kubernetes used for?',
 'How does Kubernetes manage containers?',
 'What is Docker and how does it work?']

In [22]:
print("Loading search queries...\n")

queries_df = pd.read_csv("queries.csv")
queries = queries_df["query"].tolist()

print("Total queries loaded:", len(queries))
print("Sample queries:", queries[:5], "\n")

Loading search queries...

Total queries loaded: 77
Sample queries: ['How do you build a video sharing app using Next.js?', 'What are closures in JavaScript?', 'How do regular expressions work in programming?', 'What is integration testing in software development?', 'How do you contribute to open source projects?'] 



In [23]:
print("Mapping queries to videos...\n")

mapping = []

stopwords = {
    "how", "what", "is", "the", "a", "an", "to", "can", "i", "in", "of", "for",
    "do", "does", "did", "you", "your", "with", "on", "and", "it", "this",
    "learn", "english", "scene", "dialogue", "conversation", "movie",
    "series", "tv"
}

for query in queries:
    q = query.lower()

    # remove punctuation
    q = re.sub(r"[^\w\s]", "", q)

    words = [w for w in q.split() if w not in stopwords]

    best_match = None
    best_count = 0

    for _, row in df.iterrows():
        text = (str(row["title"]) + " " + str(row["transcript"])).lower()

        count = sum(word in text for word in words)

        if count > best_count:
            best_count = count
            best_match = row["video_id"]

    video_id = best_match
    mapping.append((query, video_id))

Mapping queries to videos...



In [30]:
mapping_df = pd.DataFrame(mapping, columns=["query", "relevant_video_id"])

mapping_df.to_csv("query_video_map.csv", index=False)

print("Query -> Video mapping completed\n")

print("\nSample mappings:\n")
print(mapping_df.head())

Query -> Video mapping completed


Sample mappings:

                                               query relevant_video_id
0  How do you build a video sharing app using Nex...       IBTx5aGj-6U
1                   What are closures in JavaScript?       U6-RekkuORI
2    How do regular expressions work in programming?       gmuTjeQUbTM
3  What is integration testing in software develo...       s950xhRvqYQ
4     How do you contribute to open source projects?       cw02dMpWStI
